# Assignment #4: Soft Computer, Reworked with Hand-Tagged Semantics

reworking my first assignment (the soft computer poetry generator) to use our class's hand-tagged semantic corpus. instead of picking from small hardcoded vocabulary lists i wrote myself, each line now pulls from the nearest neighbors of a target vector in the shared tag-space. the structural scaffolding of the original poem stays, but the vocabulary comes from the class's collective sense of which words feel organic, warm, sharp, emotive, etc.

In [2]:
import json
import random
from collections import defaultdict, Counter
from simpleneighbors import SimpleNeighbors

In [3]:
%pip install simpleneighbors scikit-learn numpy

Note: you may need to restart the kernel to use updated packages.


In [4]:
# loading the tagged data
lines = open("semantic-categories-2026-04-10.jsonl").readlines()

cats = defaultdict(list)
for item in lines:
    obj = json.loads(item)
    if len(obj['label']) > 0:
        cats[obj['text'].lower()].extend(obj['label'])

In [5]:
# count tags per word
cats_counted = {word: Counter(tags) for word, tags in cats.items()}

In [6]:
# build the column order for vectors
unique_features = set()
for val in cats.values():
    unique_features.update(val)

cols = list(unique_features)
name2idx = {name: i for i, name in enumerate(cols)}
print(cols)

['cold', 'emotive', 'round', 'outlandish', 'warm', 'youthful', 'sharp', 'organic', 'academic', 'tangible']


In [7]:
# vectorize every word
cats_vectorized = {}
for word, counts in cats_counted.items():
    cats_vectorized[word] = [counts[feat] for feat in cols]

In [8]:
# build the nearest-neighbor lookup
lookup = SimpleNeighbors(len(cols), metric="angular")
for word, vec in cats_vectorized.items():
    lookup.add_one(word, vec)
lookup.build()

In [9]:
# helper to build target vectors by tag name
def make_vector(**kwargs):
    vec = [0] * len(cols)
    for k, v in kwargs.items():
        vec[name2idx[k]] = v
    return vec

## nearest neighbors only
The main poem generator, each slot has a target vector describing the "vibe" I want - material aims for organic + tangible, power aims for warm + organic, inhabitant aims for emotive + warm. The generator picks a random word from the 20 nearest neighbors of each target.

In [10]:
# generate poem
stanza_count = 7

for i in range(stanza_count):
    material   = random.choice(lookup.nearest(make_vector(organic=2, tangible=2), n=20))
    structure  = random.choice(lookup.nearest(make_vector(round=2, warm=1), n=20))
    power      = random.choice(lookup.nearest(make_vector(warm=3, organic=1), n=20))
    interface  = random.choice(lookup.nearest(make_vector(tangible=2, sharp=1), n=20))
    location   = random.choice(lookup.nearest(make_vector(warm=2, round=2), n=20))
    inhabitant = random.choice(lookup.nearest(make_vector(emotive=3, warm=1), n=20))
    promise    = random.choice(lookup.nearest(make_vector(emotive=2, youthful=1), n=20))

    print()
    print(f"a computer of {material}")
    print(f"     bound by {structure}")
    print(f"          powered by {power}")
    print(f"                with {interface}")
    print(f"                     resting among {location}")
    print(f"                          inhabited by {inhabitant}")
    print(f"                               and it offers {promise}.")


a computer of necessity
     bound by okay
          powered by simmering
                with steps
                     resting among lamp
                          inhabited by support
                               and it offers fame.

a computer of men singers
     bound by salon
          powered by sunset
                with dock
                     resting among workmanship
                          inhabited by amazing
                               and it offers girlish.

a computer of trace
     bound by harmonious
          powered by praise
                with speakers
                     resting among nod
                          inhabited by humble
                               and it offers beautiful.

a computer of tall
     bound by harmonious
          powered by heat
                with stitch
                     resting among everyone
                          inhabited by sorrow
                               and it offers jolly.

a computer of flutter
  

## rejection
The first version still let in words tagged cold or sharp (hunger, gunpowder, straight). This version adds a rule: refuse any word tagged cold, sharp, or academic. The soft computer isn't those things, so the vocabulary shouldn't be either.

In [11]:
# helper to check if a word has forbidden tags
def has_forbidden(word, forbidden):
    """return True if the word was tagged with any forbidden category"""
    tags = set(cats_counted.get(word, Counter()).keys())
    return bool(tags & forbidden)

def pick_soft(target_vec, forbidden, pool_size=40, max_tries=20):
    """pick a word near target_vec that has none of the forbidden tags"""
    candidates = lookup.nearest(target_vec, n=pool_size)
    for _ in range(max_tries):
        word = random.choice(candidates)
        if not has_forbidden(word, forbidden):
            return word
    # if can't find a clean one, return the last candidate anyway
    return word

In [12]:
# generate, refusing cold/sharp/academic words
forbidden = {"cold", "sharp", "academic"}

stanza_count = 7

for i in range(stanza_count):
    material   = pick_soft(make_vector(organic=2, tangible=2), forbidden)
    structure  = pick_soft(make_vector(round=2, warm=1), forbidden)
    power      = pick_soft(make_vector(warm=3, organic=1), forbidden)
    interface  = pick_soft(make_vector(tangible=2), forbidden)
    location   = pick_soft(make_vector(warm=2, round=2), forbidden)
    inhabitant = pick_soft(make_vector(emotive=3, warm=1), forbidden)
    promise    = pick_soft(make_vector(emotive=2, youthful=1), forbidden)

    print()
    print(f"a computer of {material}")
    print(f"     bound by {structure}")
    print(f"          powered by {power}")
    print(f"                with {interface}")
    print(f"                     resting among {location}")
    print(f"                          inhabited by {inhabitant}")
    print(f"                               and it offers {promise}.")


a computer of woman
     bound by ease
          powered by parent
                with furniture
                     resting among aura
                          inhabited by sorrow
                               and it offers true loves.

a computer of necessity
     bound by camera
          powered by parent
                with temples
                     resting among pockets
                          inhabited by plays
                               and it offers near.

a computer of joint
     bound by worlds
          powered by summer
                with proficient
                     resting among kind
                          inhabited by daily
                               and it offers true loves.

a computer of glass
     bound by camera
          powered by asleep
                with teaspoon
                     resting among breast
                          inhabited by personal
                               and it offers _statirical_.

a computer of finger
 

## tag-descriptor
Each word is printed with its top two tags in parentheses, making the hand-tagging visible in the output itself. More of a formal study than a poem.

In [13]:
# helper to read top tags for a word
def top_tags(word, n=2):
    """return the n most-tagged categories for a word, as a list"""
    counts = cats_counted.get(word, Counter())
    return [tag for tag, _ in counts.most_common(n)]

In [14]:
# generate with tag descriptors
stanza_count = 7

for i in range(stanza_count):
    material   = random.choice(lookup.nearest(make_vector(organic=2, tangible=2), n=20))
    structure  = random.choice(lookup.nearest(make_vector(round=2, warm=1), n=20))
    power      = random.choice(lookup.nearest(make_vector(warm=3, organic=1), n=20))
    interface  = random.choice(lookup.nearest(make_vector(tangible=2, sharp=1), n=20))
    location   = random.choice(lookup.nearest(make_vector(warm=2, round=2), n=20))
    inhabitant = random.choice(lookup.nearest(make_vector(emotive=3, warm=1), n=20))
    promise    = random.choice(lookup.nearest(make_vector(emotive=2, youthful=1), n=20))

    def describe(word):
        tags = top_tags(word, 2)
        return f"{word} ({', '.join(tags)})" if tags else word

    print()
    print(f"a computer of {describe(material)}")
    print(f"     bound by {describe(structure)}")
    print(f"          powered by {describe(power)}")
    print(f"                with {describe(interface)}")
    print(f"                     resting among {describe(location)}")
    print(f"                          inhabited by {describe(inhabitant)}")
    print(f"                               and it offers {describe(promise)}.")


a computer of tall (organic, tangible)
     bound by something (round)
          powered by glow (warm, organic)
                with pen (tangible, sharp)
                     resting among kiln (warm, round)
                          inhabited by dread (emotive)
                               and it offers beautiful (emotive, youthful).

a computer of posts (organic, tangible)
     bound by character (round)
          powered by nightingale (warm, organic)
                with real (tangible, organic)
                     resting among coats (warm, round)
                          inhabited by amazing (emotive)
                               and it offers fame (emotive, youthful).

a computer of chestnut (organic, tangible)
     bound by character (round)
          powered by red (warm, organic)
                with hunger (tangible, sharp)
                     resting among workmanship (round, warm)
                          inhabited by support (emotive)
                          